In [25]:
import os, pickle

from physics.simulation import mcfm, msq
from physics.hzz import zpair, zz4l
from datasets import balanced
from models import carl

import numpy as np
import pandas as pd
import matplotlib, matplotlib.pyplot as plt
import hist

import torch
from torch.utils.data import Dataset
import lightning

In [26]:
torch.set_float32_matmul_precision('highest')

matplotlib.use("pgf")
matplotlib.rcParams.update({
    "pgf.texsystem": "lualatex",
    "text.usetex": True,
    "pgf.rcfonts": False,  # Don't override with default matplotlib fonts
    "pgf.preamble": "", 
    #     "\\usepackage{fontspec}"
    #     "\\setmainfont{Computer Modern Roman}"  # or Computer Modern if installed
})


In [6]:
OUTPUT_DIR = 'run/carl/'
SCALER_FILE_X = 'scaler.pkl'
SCALER_FILE_Y = None
CHECKPOINT_DIR = 'checkpoints'
SAMPLE_DIR = 'data/'

EPOCH = '153'           # Epoch and loss of the checkpoint
VAL_LOSS = 0.68
VERSION = 1             # Lightning logs version number

CHECKPOINT = f'checkpoint-carl-epoch={EPOCH}-val_loss={VAL_LOSS}.ckpt'
LIGHTNING_DIR = f'lightning_logs/version_{VERSION}'

SAMPLE_SIZE = 2500000

FEATURES=['l1_pt', 'l1_eta', 'l1_phi', 'l1_energy',
          'l2_pt', 'l2_eta', 'l2_phi', 'l2_energy',
          'l3_pt', 'l3_eta', 'l3_phi', 'l3_energy',
          'l4_pt', 'l4_eta', 'l4_phi', 'l4_energy']

BATCH_SIZE = 256
SEED = 42

In [3]:
events_numerator = mcfm.from_csv(cross_section=14.482054, file_path=os.path.join(SAMPLE_DIR, 'qqZZ2e2m/events.csv'))
events_denominator = mcfm.from_csv(cross_section=1.5569109, file_path=os.path.join(SAMPLE_DIR, 'ggZZ2e2m_sbi/events.csv'))

z_cand = zpair.ZPairCandidate(algorithm='leastsquare')
z_masses = zpair.ZPairMassWindow(z1=(70,115), z2=(70,115))
angles = zz4l.AngularVariables()
four_lepton_vars = zz4l.FourLeptonSystem()
lepton_momenta = zz4l.LeptonMomenta()

events_numerator = events_numerator.calculate(z_cand).filter(z_masses).calculate(angles).calculate(four_lepton_vars).calculate(lepton_momenta)
events_denominator = events_denominator.calculate(z_cand).filter(z_masses).calculate(angles).calculate(four_lepton_vars).calculate(lepton_momenta)

events_num_train, events_num_val = events_numerator.shuffle(random_state=SEED).split(train_size=0.5, val_size=0.5)
events_den_train, events_den_val = events_denominator.shuffle(random_state=SEED).split(train_size=0.5, val_size=0.5)

training_data = balanced.BalancedDataset(events_num_train, events_den_train, FEATURES, SAMPLE_SIZE, SEED)
validation_data = balanced.BalancedDataset(events_num_val, events_den_val, FEATURES, SAMPLE_SIZE, SEED)

print(validation_data.X.shape)
print(validation_data.s.shape)

if SCALER_FILE_X is not None:
    with open(os.path.join(OUTPUT_DIR, SCALER_FILE_X), 'rb') as f:
        scaler_X = pickle.load(f)
    
    training_data.X = scaler_X.transform(training_data.X)
    validation_data.X = scaler_X.transform(validation_data.X)

(5000000, 16)
(5000000,)


In [8]:
loaded_model = carl.CARL.load_from_checkpoint(os.path.join(OUTPUT_DIR, CHECKPOINT_DIR, CHECKPOINT))

dl_train = torch.utils.data.DataLoader(training_data, batch_size=BATCH_SIZE)
dl_val = torch.utils.data.DataLoader(validation_data, batch_size=BATCH_SIZE)

trainer = lightning.Trainer(accelerator='gpu', devices=1)

predictions_train = torch.concatenate(trainer.predict(loaded_model, dataloaders=[dl_train]), axis=0).detach().numpy()

predictions_val = torch.concatenate(trainer.predict(loaded_model, dataloaders=[dl_val]), axis=0).detach().numpy()

ratios_train_pred = predictions_train/(1-predictions_train)

ratios_val_pred = predictions_val/(1-predictions_val)

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA H200') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]


Predicting: |                                                                                                 …

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]


Predicting: |                                                                                                 …

In [11]:
targets_val = validation_data.s

In [44]:
bins = 60
bounds = [0,1]
step_size = (bounds[1]-bounds[0])/bins

p = predictions_val
t = targets_val

pred_binned = [p[(p > bounds[0]+step_size*i) & (p <= bounds[0]+step_size*(i+1))] for i in range(bins)]
targets_binned = [t[(p > bounds[0]+step_size*i) & (p <= bounds[0]+step_size*(i+1))] for i in range(bins)]

sig_per_bin = np.array([np.sum(targets_binned[i]==1.0)/len(targets_binned[i]) for i in range(bins)])
bkg_per_bin = np.array([np.sum(targets_binned[i]==0.0)/len(targets_binned[i]) for i in range(bins)])

sig_dev = np.sqrt(sig_per_bin)
bkg_dev = np.sqrt(bkg_per_bin)

s_true = sig_per_bin/(sig_per_bin+bkg_per_bin)

pred_ratios_avg = [np.mean(pred_binned[i]) for i in range(bins)]

centers = [bounds[0]+(i+1/2)*step_size for i in range(bins)]

fig, (ax1, ax2) = plt.subplots(2,1, gridspec_kw={'height_ratios': [3,1]}, figsize=(4,5), sharex=True)

ax1.set_xticklabels([])

ax1.errorbar(centers, centers, color='black', linestyle='--', label='$\mathrm{MC}$ $\mathrm{estimate}$')
ax1.errorbar(centers, s_true, yerr=0., color='royalblue', marker='o', markersize=4, linestyle='none', label='$\mathrm{NSBI}$ $\mathrm{estimate}$')

ax1.set_ylim(*bounds)
ax1.set_ylabel('$s(x) = \\frac{p_{q\\bar{q}}(x)}{ p_{q\\bar{q}}(x) + p_{gg}(x) }$')

ax1.legend(frameon=False)

ax2.errorbar(centers, np.array(centers)-np.array(centers), color='black', linestyle='--')
ax2.errorbar(centers, np.array(s_true)-np.array(centers), yerr=0., color='royalblue', marker='o', markersize=4, linestyle='none')

ax2.set_xlim(*bounds)
ax2.set_xlabel('$\\hat{s}(x)$')
ax2.set_ylabel('$\\mathrm{Residual}$')
ax2.set_ylim(-0.5, 0.5)

plt.tight_layout()
plt.savefig('carl_calibration.pdf', bbox_inches='tight')
plt.close()

/tmp/ipykernel_2991722/2767043937.py:11: RuntimeWarning: invalid value encountered in scalar divide
  sig_per_bin = np.array([np.sum(targets_binned[i]==1.0)/len(targets_binned[i]) for i in range(bins)])
/tmp/ipykernel_2991722/2767043937.py:12: RuntimeWarning: invalid value encountered in scalar divide
  bkg_per_bin = np.array([np.sum(targets_binned[i]==0.0)/len(targets_binned[i]) for i in range(bins)])
/opt/conda/lib/python3.10/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/conda/lib/python3.10/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)
